# Upload a video and test the existing models

**Run the two code cells individually:** CPU upload → T4 inference → download directly from Drive. Use the same Google Drive as the previous training runs; the pretrained assets must already exist there.

The default processes the entire uploaded video. Cell 2 saves and verifies the output in private Drive, then releases the GPU. Step 3 opens Drive; no runtime switch or reconnection is needed. If Drive preservation fails, the GPU stays connected so you can save the remaining runtime files.

This uses the existing LCD9, speed11 and RPM9 models with fast LCD-based RPM positioning. **No retraining; raw readings remain unvalidated.** The upright LCD corner assumption and unmasked speed input retain the known diagnostic limitations. Videos and weights are never published.


In [ ]:
#@title 1. Upload a video on CPU and save it to Drive
# Run the cells individually. Use the same Drive as your previous model runs.
# Keep Hardware accelerator = None until the upload is fully saved.
START_SECONDS = 0.0 #@param {type:"number"}
MAX_SECONDS = 0.0 #@param {type:"number"}
FORCE_NEW_UPLOAD = False #@param {type:"boolean"}
# MAX_SECONDS=0 processes the entire remaining video. No old128-frame limit.
# Public code is pinned; videos and weights remain in your private Drive.
from pathlib import Path
import hashlib, importlib.util, os, urllib.request, uuid
CODE_COMMIT = '63a1d14fb441ea8f19cc0a22fcda2e206652da58'
REPOSITORY = 'sainadh812/triumph-speedometer-dataset'
EXPECTED_FILES = {'colab_video_helpers.py': {'bytes': 41443, 'sha256': 'f48f161fcf9ed2c1c86754698f41554090d3fb9ea671c693322819be4e298bfe'}, 'evaluate_uploaded_video.py': {'bytes': 28779, 'sha256': '5ca4b7b1c59be091b23f999bc8e571fb3275eea37792abb67b3e147b11cb33ef'}, 'evaluate_lcd_speed_rpm_batched.py': {'bytes': 55284, 'sha256': '0c5df9a13e5cd904d76339a74de8d84675965041ee166eacf94e933214947df3'}, 'benchmark_lcd_positioning.py': {'bytes': 21561, 'sha256': '68c299ada88ff433fea7e57ac379823ee9d6848d61f0605556642ecc3c5957e0'}}
RUNTIME_DIR = Path('/content/speedometer_upload_runtime') / CODE_COMMIT
RUNTIME_DIR.mkdir(parents=True, exist_ok=True)
for name, metadata in EXPECTED_FILES.items():
    destination = RUNTIME_DIR / name
    if destination.is_file():
        data = destination.read_bytes()
    else:
        url = 'https://raw.githubusercontent.com/' + REPOSITORY + '/' + CODE_COMMIT + '/learned_lcd_v1/upload_video_test_v1/' + name
        try:
            with urllib.request.urlopen(url, timeout=60) as response:
                data = response.read()
        except Exception as error:
            raise RuntimeError('Could not fetch pinned public runtime code: ' + name) from error
    if len(data) != metadata['bytes'] or hashlib.sha256(data).hexdigest() != metadata['sha256']:
        raise ValueError('Runtime file differs from the reviewed release: ' + name)
    if not destination.exists():
        temporary = destination.with_name(name + '.staged-' + uuid.uuid4().hex)
        temporary.write_bytes(data)
        os.replace(temporary, destination)
specification = importlib.util.spec_from_file_location('speedometer_upload_helpers', RUNTIME_DIR / 'colab_video_helpers.py')
helpers = importlib.util.module_from_spec(specification)
specification.loader.exec_module(helpers)  # Import-only; no model work.
helpers.require_cpu()
from google.colab import drive
if not Path('/content/drive/MyDrive').is_dir():
    drive.mount('/content/drive', timeout_ms=300000)

job = helpers.upload_video(START_SECONDS, MAX_SECONDS, FORCE_NEW_UPLOAD)


In [ ]:
#@title 2. Switch to T4 GPU, then run the saved video
# Runtime > Change runtime type > T4 GPU. Reconnect and run only this cell.
# Existing LCD9 + speed11 + RPM9 weights. Fast LCD positioning; no training.
# Large RPM label plus raw speed. Old-model readings remain unvalidated.
AUTO_DISCONNECT = True #@param {type:"boolean"}
inference_started = False
try:
    # Public code is pinned; videos and weights remain in your private Drive.
    from pathlib import Path
    import hashlib, importlib.util, os, urllib.request, uuid
    CODE_COMMIT = '63a1d14fb441ea8f19cc0a22fcda2e206652da58'
    REPOSITORY = 'sainadh812/triumph-speedometer-dataset'
    EXPECTED_FILES = {'colab_video_helpers.py': {'bytes': 41443, 'sha256': 'f48f161fcf9ed2c1c86754698f41554090d3fb9ea671c693322819be4e298bfe'}, 'evaluate_uploaded_video.py': {'bytes': 28779, 'sha256': '5ca4b7b1c59be091b23f999bc8e571fb3275eea37792abb67b3e147b11cb33ef'}, 'evaluate_lcd_speed_rpm_batched.py': {'bytes': 55284, 'sha256': '0c5df9a13e5cd904d76339a74de8d84675965041ee166eacf94e933214947df3'}, 'benchmark_lcd_positioning.py': {'bytes': 21561, 'sha256': '68c299ada88ff433fea7e57ac379823ee9d6848d61f0605556642ecc3c5957e0'}}
    RUNTIME_DIR = Path('/content/speedometer_upload_runtime') / CODE_COMMIT
    RUNTIME_DIR.mkdir(parents=True, exist_ok=True)
    for name, metadata in EXPECTED_FILES.items():
        destination = RUNTIME_DIR / name
        if destination.is_file():
            data = destination.read_bytes()
        else:
            url = 'https://raw.githubusercontent.com/' + REPOSITORY + '/' + CODE_COMMIT + '/learned_lcd_v1/upload_video_test_v1/' + name
            try:
                with urllib.request.urlopen(url, timeout=60) as response:
                    data = response.read()
            except Exception as error:
                raise RuntimeError('Could not fetch pinned public runtime code: ' + name) from error
        if len(data) != metadata['bytes'] or hashlib.sha256(data).hexdigest() != metadata['sha256']:
            raise ValueError('Runtime file differs from the reviewed release: ' + name)
        if not destination.exists():
            temporary = destination.with_name(name + '.staged-' + uuid.uuid4().hex)
            temporary.write_bytes(data)
            os.replace(temporary, destination)
    specification = importlib.util.spec_from_file_location('speedometer_upload_helpers', RUNTIME_DIR / 'colab_video_helpers.py')
    helpers = importlib.util.module_from_spec(specification)
    specification.loader.exec_module(helpers)  # Import-only; no model work.
    helpers.require_gpu()
    from google.colab import drive
    if not Path('/content/drive/MyDrive').is_dir():
        drive.mount('/content/drive', timeout_ms=300000)

    inference_started = True
    receipt = helpers.run_uploaded_video(RUNTIME_DIR, auto_disconnect=AUTO_DISCONNECT)
except BaseException as setup_error:
    if not inference_started and AUTO_DISCONNECT:
        print("Setup failed before inference; no runtime result was produced:", repr(setup_error), flush=True)
        try:
            import shutil, subprocess
            executable = shutil.which("nvidia-smi")
            gpu = subprocess.run([executable, "-L"], capture_output=True, text=True, timeout=15) if executable else None
            if gpu is not None and gpu.returncode == 0 and "GPU" in gpu.stdout:
                from google.colab import runtime
                runtime.unassign()
        except Exception as release_error:
            print("Use Runtime > Disconnect and delete runtime if still connected:", repr(release_error))
    raise


### 3. Download the finished video directly from Google Drive

**No cell to run, no CPU switch, and no runtime connection needed.**

Wait until cell 2 prints **Drive backup verified**. Cell 2 already saves the labeled MP4, predictions and timings to Drive and verifies their hashes before automatic GPU disconnect. Changing or deleting the Colab runtime does not delete these saved Drive files.

1. Use **Open this private result folder in Drive** in cell 2's output. If that output is unavailable, [find the browser_upload_v1 folder in Drive](https://drive.google.com/drive/search?q=browser_upload_v1).
2. Use the same Google account as the training runs. Open `jobs` → your video's `video_...` folder → `runs` → the completed `run_...` folder → `evaluation`.
3. Right-click **annotated_fast_lcd_speed_rpm.mp4** and choose **Download**.

Full location: `My Drive/SpeedometerTraining/video_tests/browser_upload_v1/jobs/<job_id>/runs/<run_id>/evaluation/annotated_fast_lcd_speed_rpm.mp4`

Download the completed run matching your uploaded video. A run ending in `_incomplete` is a failed partial run. If cell 2 is still processing, wait; a download step cannot finish the video. If you turned **AUTO_DISCONNECT** off, disconnect and delete the GPU runtime after **Drive backup verified** appears. The Drive download continues independently of Colab.
